In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np

# Clustering using K-Means
> https://en.wikipedia.org/wiki/K-means_clustering

Given a ndarray X with n indices and $x_0$ to $x_{n-1}$ observations, where each observation $x_i$ is an 1-dimensional array, k-means clustering partitions the observations into buckets or clusters where observations belonging to the same cluster have their means closer to each other. The $k-$ in k-means clustering determines the number of buckets we want to generate from the given set of observations. The value of $k$ should be always *less than or equal to* $n$.

### K-Means Algorithm
1. Specify the value of $k$ such that $k<=n$, where $n$ is the number of observations in the data set
2. Randomly select $k$ observations from the data set as *centroids*. Centroid are the central point for a cluster
3. Using a distance function, such as euclidean distance measure the distance of all observations $x_i$ from all the centroids
  - Euclidean Distance: 
  $$
  d(p, q) = \sqrt{\sum_{0}^{n-1}(q_i - p_i)^2}
  $$
4. Assign each observation to the cluster having the minimum distance with its centroid
5. Compute new centroids for each cluster by taking the average of its observations
6. Repeat steps $3-5$ until the centroid are the same in two consecutive runs

### Dataset
The Zoo dataset can be used to understand K-Means. The dataset can be acquired from: https://archive.ics.uci.edu/ml/datasets/zoo

In [ ]:
# Load the dataset
data = pd.read_csv('./data/zoo.data', header=None, index_col=None)
data.head()


In [ ]:
# Load the data column names
columns = pd.read_csv(
    './data/zoo.names.short.csv', header=None)
list(columns.to_numpy().flatten())


In [ ]:
# Set the column names
data.columns = list(columns.to_numpy().flatten())
data.head()

In [ ]:
# In pandas, isna() and isnull() are functionally equivalent and both detect missing values (NaN or None).
print('number of na: {0} and null: {1}'.format(data.isna().sum(axis=0).sum(), data.isnull().sum(axis=1).sum()))

In [ ]:
data.describe()

In [ ]:
data.columns

In [ ]:
# Visualizing relationship between all features and label type
fig, ax = plt.subplots(4,4)
fig.set_size_inches(10,10)
ax = ax.flatten()
# ....difference({'animalname', 'type'}) excludes these two columns for the result that is produced
for i, c in enumerate(set(data.columns).difference({'animal name', 'type'})):  
    ax[i].scatter(pd.Categorical(data['type']), data[c])
    ax[i].set_title('type-' + c)
    ax[i].set_xticks([])


In [ ]:
#Calculating the distance between all points (rows) to a centroid
# returns a vector of distances between each point in X and the centroid c
def distance(X, c):
    return np.sqrt(np.sum(np.square(X - c), axis=1))


In [ ]:
# Scale all features/columns
# the feature values are centered around 0 and have a standard deviation of 1
scaledData = data.loc[:, list(set(data.columns).difference({'animal name', 'type'}))].apply(
    lambda s: (s - s.mean())/s.std(), axis=0)


In [ ]:
features = scaledData.to_numpy()
features.shape

In [ ]:
# Set the number of clusters. CHANGE the value for k and see what happens to accuracy
k = 7

# randomly choose k points as centroids
cIndices = np.random.choice(range(len(features)), k )
C = features[cIndices]

In [ ]:
# Will store a map of the closest centroid index and distance to a point
closestCentroid = {}

In [ ]:
# Identifies the centroid that is closest to a point
def measureCentroidDistance(ci, dist):
    for i, d in enumerate(dist):
        # if for a point a centroid is not defined
        if not closestCentroid.get(i):
            # We set the current centroid as its closest
            closestCentroid[i] = (ci, d)
        # If a point already has a centroid defined
        else:
            # We check if the distance of the current centroid is smaller than the centroid already assigned to the point
            if d < closestCentroid[i][1]:
                # If yes we update the centroid and distance for the point
                closestCentroid[i] = (ci, d)


In [ ]:
for ci, ri in enumerate(cIndices):
    # sets the closest centroid of the points choosen to be centroid as self with distance as 0
    closestCentroid[ri] = (ci, 0.0)

closestCentroid


In [ ]:
# iterate through all centroids
for i, c in enumerate(C):
    # Calculate the distance of all point to a centroid point
    dist = distance(features, c)
    # Identify for each point which will be the closest centroid and its distance
    measureCentroidDistance(i, dist)
    


In [ ]:
# Sanity check. Checking the number of points assigned to each centroid in the first iteration
centroidCount = {}
for c in list(map(lambda x: x[0], closestCentroid.values())):
    if centroidCount.get(c):
        centroidCount[c] += 1
    else:
        centroidCount[c] = 1

centroidCount


In [ ]:
# Sort the closest centroid map by key (point index)
closestCentroid = dict(
    sorted(closestCentroid.items(), key=lambda item: item[0]))


In [ ]:
closestCentroid

In [ ]:
# We will not repeat the process by choosing new centroid. The new centroid will be the average of points assigned to each cluster
grouped = scaledData.groupby(
    list(map(lambda x: x[0], closestCentroid.values())))

_C = grouped.mean().to_numpy()


In [ ]:
# We continue the process until the allocation of points to a cluster doesn't change

while ~np.all(C == _C):
    print('shape of C: {0} and _C: {1}'.format(C.shape, _C.shape))
    closestCentroid = {}
    # Update the centroids
    C = _C
    for i, c in enumerate(C):
        dist = distance(features, c)
        measureCentroidDistance(i, dist)
    
    grouped = scaledData.groupby(
        list(map(lambda x: x[0], closestCentroid.values())))
    _C = grouped.mean().to_numpy()

    # If the re-assignment resulted in less number of centroids than the specified 'k', we add dummy centroids
    dummyCentroidIndices = np.random.choice(
        range(C.shape[0]), C.shape[0] - _C.shape[0])
    _C = np.concatenate((_C, C[dummyCentroidIndices]), axis=0)


In [ ]:
# Setting the estimated cluster ids as indices of the original data
data.index = list(map(lambda x: x[0]+1, closestCentroid.values()))

Set of animals:
```
        1 (41) aardvark, antelope, bear, boar, buffalo, calf,
                  cavy, cheetah, deer, dolphin, elephant,
                  fruitbat, giraffe, girl, goat, gorilla, hamster,
                  hare, leopard, lion, lynx, mink, mole, mongoose,
                  opossum, oryx, platypus, polecat, pony,
                  porpoise, puma, pussycat, raccoon, reindeer,
                  seal, sealion, squirrel, vampire, vole, wallaby,wolf
           2 (20) chicken, crow, dove, duck, flamingo, gull, hawk,
                  kiwi, lark, ostrich, parakeet, penguin, pheasant,
                  rhea, skimmer, skua, sparrow, swan, vulture, wren
           3 (5)  pitviper, seasnake, slowworm, tortoise, tuatara 
           4 (13) bass, carp, catfish, chub, dogfish, haddock,
                  herring, pike, piranha, seahorse, sole, stingray, tuna
           5 (4)  frog, frog, newt, toad 
           6 (8)  flea, gnat, honeybee, housefly, ladybird, moth, termite, wasp
           7 (10) clam, crab, crayfish, lobster, octopus,
                  scorpion, seawasp, slug, starfish, worm```

In [ ]:
data.head(100)

In [ ]:
#  Grouping the 'animal name' column by the 'type' column (actual cluster labels from the dataset)
# and counts the number of animals in each type.
data['animal name'].groupby(data.type).count()

In [ ]:
# Checking how many values(animals) are in cluster 2
data.loc[2]

In [ ]:
# Grouping the 'animal name' column by the DataFrame index (which was set to estimated cluster ids from K-Means)
# and counts the number of animals assigned to each predicted cluster.
data['animal name'].groupby(level=0).count()